In [3]:
from heapq import heappop, heappush
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from utils import ucs, gbfs, astar, haversine_km, load_openflights

ROOT = Path.cwd().parent
DATA = ROOT / "all_datasets"
RESULTS = ROOT / "results" / "lab02"
RESULTS.mkdir(parents=True, exist_ok=True)

_, _, coords, graph, _ = load_openflights(DATA / "lab01-03_airports.csv", DATA / "lab01-03_routes.csv")
queries = pd.read_csv("utils/search_queries.csv").head(3)
display(queries)

,query_id,start,goal
0,Q1,KUL,KEF
1,Q2,SIN,EZE
2,Q3,NRT,CPT


In [4]:
def report(name, query, result):
    print(f"{query.query_id} {name:<5} {result.path_cost_km:>9,.1f} km {result.expanded:>5} expanded {'-> '.join(result.path)}")
    
baseline = {}
for query in queries.itertuples():
    baseline[query.query_id] = ucs(graph, query.start, query.goal)
    report("UCS", query, baseline[query.query_id])

Q1 UCS    11,461.3 km  1851 expanded KUL-> HKT-> LED-> HEL-> KEF
Q2 UCS    17,817.4 km  3002 expanded SIN-> JNB-> GRU-> EZE
Q3 UCS    14,813.3 km  3042 expanded NRT-> HKG-> MRU-> CPT


In [6]:
def heuristic_to(goal, weight=1.0):
    """h(airport) = weight x great-circle distance from the airport to the goal."""
    def h(airport):
        return weight * haversine_km(coords[airport]["latitude"], coords[airport]["longitude"],
        coords[goal]["latitude"], coords[goal]["longitude"])
        return h

informed = {}
for query in queries.itertuples():
    h = heuristic_to(query.goal)
    informed[query.query_id, "GBFS"] = gbfs(graph, query.start, query.goal, h)
    informed[query.query_id, "A*"] = astar(graph, query.start, query.goal, h)
    report("GBFS", query, informed[query.query_id, "GBFS"])
    report("A*", query, informed[query.query_id, "A*"])


Q1 GBFS   12,274.3 km   284 expanded KUL-> AMS-> KEF
Q1 A*     11,461.3 km  1851 expanded KUL-> HKT-> LED-> HEL-> KEF
Q2 GBFS   18,772.0 km   233 expanded SIN-> SYD-> SCL-> EZE
Q2 A*     17,817.4 km  3002 expanded SIN-> JNB-> GRU-> EZE
Q3 GBFS   15,622.8 km   173 expanded NRT-> DXB-> CPT
Q3 A*     14,813.3 km  3042 expanded NRT-> HKG-> MRU-> CPT


In [15]:
def reverse(graph):
    reversed_graph = {airport: [] for airport in graph}
    for airport, neighbours in graph.items():
        for neighbour, cost in neighbours:
            reversed_graph[neighbour].append((airport, cost))
    return reversed_graph
    
def cost_to_all(graph, start):
    """Dijkstra: cheapest cost from start to every reachable state."""
    best = {start: 0.0}
    frontier = [(0.0, start)]
    while frontier:
        cost, state = heappop(frontier)
        if cost > best[state]: 
            continue
        for neighbour, edge in graph[state]:
            if cost + edge < best.get(neighbour, np.inf):
                best[neighbour] = cost + edge
                heappush(frontier, (cost + edge, neighbour))
                return best

reversed_graph = reverse(graph)
checks = []
for query in queries.itertuples():
    exact = cost_to_all(reversed_graph, query.goal) # exact cost-to-go for every airport
    h = heuristic_to(query.goal)
    airports = sorted((a for a in exact if a != query.goal), key=exact.get) # nearest first
    for airport in [airports[i] for i in np.linspace(0, len(airports) - 1, 10, dtype=int)]: 
        checks.append({"query_id": query.query_id, "airport": airport, "goal": query.goal, 
                       "haversine_km": h(airport), "shortest_route_km": exact[airport]})
        
check = pd.DataFrame(checks)
check["slack_km"] = check.shortest_route_km - check.haversine_km
check["admissible"] = check.slack_km >= -1e-6
check.to_csv(RESULTS / "heuristic_check.csv", index=False)
print("Admissibility violations:", int((~check.admissible).sum()), "of", len(check))
display(check.head(10).round(1))


TypeError: 'NoneType' object is not callable